In [ ]:
# パッケージのクローンとセットアップ
!git clone https://github.com/VOICEVOX/voicevox_core -b 0.13.2
%cd voicevox_core/


In [ ]:
# 環境構築
!mkdir -p onnxruntime
!wget -q https://github.com/microsoft/onnxruntime/releases/download/v1.10.0/onnxruntime-linux-x64-gpu-1.10.0.tgz
!tar xf onnxruntime-linux-x64-gpu-1.10.0.tgz -C onnxruntime --strip=1
!rm -f onnxruntime-linux-x64-gpu-1.10.0.tgz

!mkdir -p release
!wget -q https://github.com/VOICEVOX/voicevox_core/releases/download/0.13.2/voicevox_core-linux-x64-gpu-0.13.2.zip
!unzip -qj voicevox_core-linux-x64-gpu-0.13.2.zip -d release
!rm -f voicevox_core-linux-x64-gpu-0.13.2.zip

!mkdir -p core/lib
!cp onnxruntime/lib/* core/lib
!cp release/* core/lib

!wget -q http://downloads.sourceforge.net/open-jtalk/open_jtalk_dic_utf_8-1.11.tar.gz
!tar xf open_jtalk_dic_utf_8-1.11.tar.gz
!rm -f open_jtalk_dic_utf_8-1.11.tar.gz

!pip install -qqU .
!pip install -qqU aiohttp pytchat numpy sounddevice nest_asyncio openai ipython tiktoken requests ipywidgets


In [ ]:
import core
import asyncio
import pytchat
import random
import openai
import requests
import json
from IPython.display import display, HTML, Javascript
from ipywidgets import widgets
from base64 import b64decode
from google.colab import output
from google.colab import userdata
from google.colab import drive
import nest_asyncio
import IPython.display


In [ ]:
drive.mount('/content/drive')

file_path = '/content/drive/My Drive/AItuber/character.txt'
with open(file_path, encoding='utf-8') as f:
    character = f.read()

def create_character(character):
    return [{"role": "system", "content": character}]

messages = create_character(character)


In [ ]:
def initialize_core(use_gpu: bool, cpu_num_threads: int, openjtalk_dict: str):
    core.initialize(use_gpu, cpu_num_threads)
    core.voicevox_load_openjtalk_dict(openjtalk_dict)

initialize_core(use_gpu=True, cpu_num_threads=0, openjtalk_dict='open_jtalk_dic_utf_8-1.11')

openai.api_key = userdata.get('OPENAI_API_KEY')
deepl_api_key = userdata.get('DEEPL_API_KEY')


In [ ]:
def tts_and_save(text: str, speaker_id: int):
    wavefmt = core.voicevox_tts(text, speaker_id)
    if not wavefmt:
        return False
    try:
        with open('data.wav', 'wb') as f:
            f.write(wavefmt)
        return True
    except IOError as e:
        print(f'ファイル保存時のエラー: {e}')
        return False

def text_to_speech(text, speaker_id=20):
    if tts_and_save(text, speaker_id):
        display(IPython.display.Audio('data.wav', autoplay=True))
    else:
        print('TTSの実行に失敗しました')

def translate_to_english(text):
    if not deepl_api_key:
        return text
    try:
        params = {
            'auth_key': deepl_api_key,
            'text': text,
            'source_lang': 'JA',
            'target_lang': 'EN'
        }
        request = requests.post('https://api-free.deepl.com/v2/translate', data=params, timeout=30)
        request.raise_for_status()
        result = request.json()
        return result['translations'][0]['text']
    except Exception as e:
        print(f'DeepL翻訳エラー: {e}')
        return text

def update_text_in_frame(text):
    safe_text = json.dumps(text, ensure_ascii=False)
    javascript = f'''
    const container = document.getElementById('text-container');
    if (container) {{
      container.innerHTML = `
        <link rel="preconnect" href="https://fonts.googleapis.com">
        <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
        <link href="https://fonts.googleapis.com/css2?family=Yusei+Magic&display=swap" rel="stylesheet">
        <style>
          h1 {{
            font-family: 'Yusei Magic', sans-serif;
            color: pink;
            white-space: pre-wrap;
          }}
        </style>
        <h1 id="overlay-text"></h1>
      `;
      document.getElementById('overlay-text').textContent = {safe_text};
    }}
    '''
    display(Javascript(javascript))


In [ ]:
def fetch_AI_response(messages):
    response = openai.chat.completions.create(
        model='gpt-3.5-turbo-1106',
        max_tokens=50,
        temperature=0.2,
        messages=messages
    )
    return response

def speech_to_text(audio_data, model='whisper-1', language='ja'):
    b = b64decode(audio_data.split(',')[1])
    with open('tmp.wav', 'wb') as fw:
        fw.write(b)
    with open('tmp.wav', 'rb') as fr:
        transcription = openai.audio.transcriptions.create(
            model=model,
            file=fr,
            language=language,
            response_format='text'
        )
        return transcription

RECORD = r'''
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.srcElement.result);
  reader.readAsDataURL(blob);
});

let recorder, chunks, stream;

window.startRecording = async () => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(stream);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
};

window.stopRecording = () => {
  recorder.stop();
  stream.getTracks().forEach(track => track.stop());
  return new Promise(resolve => {
    recorder.onstop = async () => {
      const blob = new Blob(chunks);
      const text = await b2text(blob);
      resolve(text);
    };
  });
};
'''

start_button = widgets.Button(description='Start Recording')
stop_button = widgets.Button(description='Stop Recording')
output_area = widgets.Output()

def on_start_clicked(_):
    output_area.clear_output()
    with output_area:
        print('Recording started...')
    output.eval_js('startRecording()')

def on_stop_clicked_sync(_):
    asyncio.create_task(on_stop_clicked(_))

async def on_stop_clicked(_):
    with output_area:
        print('Recording stopped, transcribing...')
        audio_data = output.eval_js('stopRecording()')
        transcribed_text = speech_to_text(audio_data)
        messages.append({'role': 'user', 'content': transcribed_text})
        response = fetch_AI_response(messages)
        if response is not None:
            resp = response.choices[0].message.content.strip()
            print(resp)
            text_to_speech(resp, speaker_id=20)
            text = translate_to_english(resp)
            update_text_in_frame(text)
            messages.append({'role': 'assistant', 'content': resp})


In [ ]:
comments_queue = asyncio.Queue()
is_chat_ended = asyncio.Event()

def parse_keywords(raw_text: str):
    return [word.strip() for word in raw_text.split(',') if word.strip()]

async def get_chat(video_id, organizer_name='V太郎', end_message='終わり'):
    livechat = pytchat.create(video_id=video_id)
    print(f'ライブチャットを開始しました: {livechat.is_alive()}')
    try:
        while livechat.is_alive() and not is_chat_ended.is_set():
            try:
                chatdata = await asyncio.to_thread(livechat.get)
                if chatdata:
                    for c in chatdata.items:
                        print(f'キューに追加するメッセージ: {c.author.name}: {c.message}')
                        await comments_queue.put(c)
                        if organizer_name and end_message and c.author.name == organizer_name and c.message == end_message:
                            print('チャンネル主催者からの終了メッセージが検出されました。プロセスを終了します。')
                            is_chat_ended.set()
                            return
                else:
                    print('取得したチャットデータが空です。')
            except Exception as e:
                print(f'チャットデータの取得中にエラーが発生しました: {e}')
            await asyncio.sleep(5)
    finally:
        livechat.terminate()
        is_chat_ended.set()

async def drain_queue(queue, timeout):
    items = []
    try:
        while True:
            item = await asyncio.wait_for(queue.get(), timeout=timeout)
            items.append(item)
    except asyncio.TimeoutError:
        pass
    return items

async def process_chat_message(queue, messages, keywords):
    COMMENT_THRESHOLD = 1
    TIMEOUT = 30

    while not is_chat_ended.is_set():
        comments = await drain_queue(queue, TIMEOUT)

        if not comments:
            print('コメントが一定時間ありませんでした。')
            chat_message = 'コメントがなかったので、あなたは新しい話をするか、前回の話の続きを自然に話してください。いきなり話し始めてください。'

        elif len(comments) > COMMENT_THRESHOLD:
            print(f'多数のコメントがあります: {comments}')
            relevant_comments = [comment for comment in comments if any(keyword in comment.message for keyword in keywords)]

            if relevant_comments:
                chosen_comment = random.choice(relevant_comments)
                print(f'選択された関連するコメント: {chosen_comment.author.name}: {chosen_comment.message}')
            else:
                chosen_comment = random.choice(comments)
                print(f'ランダムに選択されたコメント: {chosen_comment.author.name}: {chosen_comment.message}')

            chat_message = chosen_comment.message

        else:
            print(f'取得したコメント: {comments}')
            chat_message = comments[0].message

        messages.append({'role': 'user', 'content': chat_message})

        response = await asyncio.to_thread(fetch_AI_response, messages)
        if response is not None:
            resp = response.choices[0].message.content.strip()
            print(resp)
            text_to_speech(resp, speaker_id=20)
            text = translate_to_english(resp)
            update_text_in_frame(text)
            messages.append({'role': 'assistant', 'content': resp})


In [ ]:
nest_asyncio.apply()
display(Javascript(RECORD))

async def main():
    global messages, comments_queue, is_chat_ended

    messages = create_character(character)
    comments_queue = asyncio.Queue()
    is_chat_ended = asyncio.Event()

    html_frame = HTML('''
    <div id="text-container" style="border:1px solid #ccc; padding:10px; width:1000px; height:200px; overflow:auto;">
    ここにテキストが表示されます。
    </div>
    ''')
    display(html_frame)

    select_mode = input('おしゃべりモード：1, 配信モード：2 -> ').strip()

    if select_mode == '1':
        start_button.on_click(on_start_clicked)
        stop_button.on_click(on_stop_clicked_sync)
        display(widgets.HBox([start_button, stop_button]), output_area)

    elif select_mode == '2':
        video_id = input('video_idを入力して: ').strip()
        important_words = input('重要な単語をカンマ区切りで教えてください: ').strip()
        organizer_name = input('終了判定に使う配信者名（不要なら空欄）: ').strip()
        end_message = input('終了メッセージ（不要なら空欄）: ').strip()

        collection_of_important_words = parse_keywords(important_words)
        organizer_name = organizer_name if organizer_name else 'V太郎'
        end_message = end_message if end_message else '終わり'

        await asyncio.gather(
            get_chat(video_id, organizer_name=organizer_name, end_message=end_message),
            process_chat_message(comments_queue, messages, collection_of_important_words)
        )

    else:
        print('1 か 2 を入力してください。')

asyncio.run(main())
